#  Text Vectorization with TF-IDF  

## 📌 Objective  

In this notebook, we will:   
✔ Load **preprocessed training, validation, and test datasets** (`X_train_split.pkl`, `X_val_split.pkl`, and `X_test_sub_cleaned_final.pkl`).  
✔ Apply **TF-IDF vectorization** to transform the text data into numerical features.  
✔ Save the transformed datasets for use in Machine Learning models.   



## 1. Import Required Libraries

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import sys
import os
from pathlib import Path
import importlib
import pandas as pd
import numpy as np
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer

### Setting Up Project Paths and Configurations

In [2]:
# Get the current notebook directory
CURRENT_DIR = Path(os.getcwd()).resolve()

# Automatically find the project root (go up 1 level)
PROJECT_ROOT = CURRENT_DIR.parents[1]

# Add project root to sys.path
sys.path.append(str(PROJECT_ROOT))

# Function to get relative paths from project root
def get_relative_path(absolute_path):
    return str(Path(absolute_path).relative_to(PROJECT_ROOT))

# Print project root directory
print(f"Project Root Directory: {PROJECT_ROOT.name}")  # Display only the root folder name

import config  # Now Python can find config.py

Project Root Directory: Data_Scientist_Rakuten_Project-main


 ## 1. Load  Data
 
We now **load the preprocessed datasets** to apply TF-IDF vectorization:  
✔ **`X_train_split.pkl`** → Training dataset (80%).  
✔ **`X_val_split.pkl`** → Validation dataset (20%).  
✔ **`X_test_sub_cleaned_final.pkl`** → Test dataset for challenge submission (no labels).  


In [3]:
# Reload config to ensure any updates are applied
importlib.reload(config)  

# Define paths for datasets
data_dir = Path(config.PROCESSED_DIR)
train_path = data_dir / "X_train_split.pkl"
val_path = data_dir / "X_val_split.pkl"
test_sub_path = data_dir / "X_test_sub_cleaned_final.pkl"  # NEW FILE

# Function to load a Pickle file safely
def load_pickle(file_path, dataset_name):
    """Loads a pickle file with error handling and basic visualization."""
    if not file_path.exists():
        print(f"[X] Error: `{dataset_name}` file not found at {file_path}")
        return None

    try:
        data = pd.read_pickle(file_path)
        print(f"[✔] Successfully loaded `{dataset_name}` | Shape: {data.shape}")

        if isinstance(data, pd.DataFrame) and not data.empty:
            display(data.head())  # Display first rows for quick verification

        return data
    except Exception as e:
        print(f"[X] Error loading `{dataset_name}`: {e}")
        return None

# Load datasets
X_train = load_pickle(train_path, "X_train_split.pkl")
X_val = load_pickle(val_path, "X_val_split.pkl")
X_test_sub = load_pickle(test_sub_path, "X_test_sub_cleaned_final.pkl") 


[✔] Successfully loaded `X_train_split.pkl` | Shape: (67932, 9)


,designation,description,text,productid,imageid,prdtypecode,prdtypecode_encoded,Label,image_name
1887,porte bebe violet rouge trois mere multifoncti...,Porte bébé Violet et rouge Trois-en-un mère mu...,porte bebe violet rouge trois mere multifoncti...,3050424970,1187504001,1320,12,Early Childhood,image_1187504001_product_3050424970.jpg
70389,jesus cahiers libre avenir,Prêtre autrement.,jesus cahiers libre avenir pretre autrement,131641431,885888766,10,0,Adult Books,image_885888766_product_131641431.jpg
59835,chambre paillasson forme coeur tapis fluffy ta...,Chambre Paillasson en forme de coeur Tapis Tap...,chambre paillasson forme coeur tapis fluffy ch...,4197486437,1313030973,1560,13,Interior Furniture and Bedding,image_1313030973_product_4197486437.jpg
23220,pcs alliage aluminium portail carter entrainem...,2pcs en alliage d&#39;aluminium Portail du car...,pcs alliage aluminium portail carter entrainem...,3929174950,1265009801,1280,7,Toys for Children,image_1265009801_product_3929174950.jpg
36107,harnais chien arnais noir anti traction gilet ...,<p><b>La description:</b></p><br /><p> Fait de...,harnais chien arnais noir anti traction gilet ...,4183293159,1313455838,2220,17,Supplies for Domestic Animals,image_1313455838_product_4183293159.jpg


[✔] Successfully loaded `X_val_split.pkl` | Shape: (16984, 9)


,designation,description,text,productid,imageid,prdtypecode,prdtypecode_encoded,Label,image_name
81432,bas filles enfants enfants collant coton bebe ...,Filles Bas Enfants Enfants Collant Coton bébé ...,bas filles enfants collant coton bebe stocking...,3898715946,1261369347,1301,10,Accessories for Children,image_1261369347_product_3898715946.jpg
44734,cosmic planete series peluche capuche couvertu...,Cosmic Planète Series en peluche avec capuche ...,cosmic planete series peluche capuche couvertu...,4205111198,1315322348,1560,13,Interior Furniture and Bedding,image_1315322348_product_4205111198.jpg
59366,dolphin robot electrique piscine fond parois l...,dolphin dolphin - robot électrique de piscine ...,dolphin robot electrique piscine fond parois l...,3894338575,1260564839,2583,23,Piscine and Spa,image_1260564839_product_3894338575.jpg
36932,haydaim pokemon noir blanc,NaN,haydaim pokemon noir blanc,155433978,911177853,1160,5,Playing Cards,image_911177853_product_155433978.jpg
69999,lot livres partitions piano bach busoni clavie...,NaN,lot livres partitions piano bach busoni clavie...,2145087508,1128429580,2403,19,Children Books and Magazines,image_1128429580_product_2145087508.jpg


[✔] Successfully loaded `X_test_sub_cleaned_final.pkl` | Shape: (13812, 6)


,designation,description,text,productid,imageid,image_name
84916,folkmanis puppets marionnette theatre mini turtle,NaN,folkmanis puppets marionnette theatre mini turtle,516376098,1019294171,image_1019294171_product_516376098.jpg
84917,porte flamme gaxix flamebringer gaxix twilight...,NaN,porte flamme gaxix flamebringer twilight dragons,133389013,1274228667,image_1274228667_product_133389013.jpg
84918,pompe filtration speck badu,NaN,pompe filtration speck badu,4128438366,1295960357,image_1295960357_product_4128438366.jpg
84919,robot piscine electrique,<p>Ce robot de piscine d&#39;un design innovan...,robot piscine electrique design innovant elega...,3929899732,1265224052,image_1265224052_product_3929899732.jpg
84920,hsm destructeur securio coupe croise,NaN,hsm destructeur securio coupe croise,152993898,940543690,image_940543690_product_152993898.jpg


## 2. Apply TF-IDF Vectorization

To transform the text data into numerical features, we apply **TF-IDF (Term Frequency - Inverse Document Frequency)** vectorization.  

###  **Key Steps**  
✔ **Fit TF-IDF on `X_train_split["text"]`** → Learns the vocabulary and importance of words.  
✔ **Transform `X_train_split`, `X_val_split`, and `X_test_sub_cleaned_final`** → Ensures consistency across datasets.  
✔ **Use `max_features=5000`** → Limits vocabulary size for efficiency while retaining key information.  
✔ **Save the TF-IDF matrix and vectorizer** for reuse in later steps.  

**Note:** `X_val_split` and `X_test_sub_cleaned_final` are **transformed with the same TF-IDF model trained on `X_train_split`** to maintain consistency.


In [4]:
# Extract text data
train_text = X_train["text"]
val_text = X_val["text"]
test_text = X_test_sub["text"]

# Initialize TF-IDF vectorizer
tfidf = TfidfVectorizer(max_features=5000)  # Limit vocabulary size for efficiency

# Fit and transform on training text
X_train_tfidf = tfidf.fit_transform(train_text)
print(f"[✔] TF-IDF fitted and transformed on X_train | Shape: {X_train_tfidf.shape}")

# Transform validation and test text using the same vectorizer
X_val_tfidf = tfidf.transform(val_text)
X_test_tfidf = tfidf.transform(test_text)
print(f"[✔] TF-IDF transformed on X_val | Shape: {X_val_tfidf.shape}")
print(f"[✔] TF-IDF transformed on X_test_sub | Shape: {X_test_tfidf.shape}")

# Display a sample of the TF-IDF X_train matrix
print(" Sample of the TF-IDF X_train Matrix:")
print(X_train_tfidf[:5, :5].toarray())  # Displaying a small portion of the matrix (5 rows, 5 columns)

# Display the corresponding words (vocabulary terms) for the sample
print("\nCorresponding feature names (terms) for the sample:")
print(tfidf.get_feature_names_out()[:5])  # Display the first 5 feature names


[✔] TF-IDF fitted and transformed on X_train | Shape: (67932, 5000)
[✔] TF-IDF transformed on X_val | Shape: (16984, 5000)
[✔] TF-IDF transformed on X_test_sub | Shape: (13812, 5000)
 Sample of the TF-IDF X_train Matrix:
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]

Corresponding feature names (terms) for the sample:
['aaa' 'aberration' 'ability' 'abord' 'abrasion']


## 4. Save TF-IDF Data  

Now that we have transformed the text data into numerical features using **TF-IDF**, we save the processed files for future use.  

### 🔹 **Saved Files & Their Purpose**  
✔ **`Xtrain_matrix.pkl`** → TF-IDF representation of the training data.  
✔ **`Xval_matrix.pkl`** → TF-IDF representation of the validation data (used for model evaluation).  
✔ **`Xtest_matrix.pkl`** → TF-IDF representation of the test data (for final submission).  
✔ **`tfidf_vectorizer.pkl`** → The trained vectorizer, ensuring we apply the same transformation later.  

These files will be used in upcoming notebooks for **ML model training, validation, and submission predictions**.  


In [5]:
# Reload config to ensure any updates are applied
importlib.reload(config)  

# Define save paths in the processed directory
save_dir = Path(config.PROCESSED_DIR)
save_dir.mkdir(parents=True, exist_ok=True)  # Ensure directory exists

# Paths for saving TF-IDF data
train_tfidf_path = save_dir / "Xtrain_matrix.pkl"
val_tfidf_path = save_dir / "Xval_matrix.pkl"  # NEW FILE (used for validation)
test_tfidf_path = save_dir / "Xtest_matrix.pkl"
vectorizer_path = save_dir / "tfidf_vectorizer.pkl"

# Save TF-IDF matrices
with open(train_tfidf_path, "wb") as f:
    pickle.dump(X_train_tfidf, f)

with open(val_tfidf_path, "wb") as f:
    pickle.dump(X_val_tfidf, f)

with open(test_tfidf_path, "wb") as f:
    pickle.dump(X_test_tfidf, f)

# Save the trained TF-IDF vectorizer
with open(vectorizer_path, "wb") as f:
    pickle.dump(tfidf, f)

# Print confirmation messages
print(f"[✔] TF-IDF training matrix saved at: {train_tfidf_path}")
print(f"[✔] TF-IDF validation matrix saved at: {val_tfidf_path}")
print(f"[✔] TF-IDF test matrix saved at: {test_tfidf_path}")
print(f"[✔] TF-IDF vectorizer saved at: {vectorizer_path}")


[✔] TF-IDF training matrix saved at: D:\Data_Science\Append_Data_Engineer_AWS_MLOPS\Data_Scientist_Rakuten_Project-main\data\processed\Xtrain_matrix.pkl
[✔] TF-IDF validation matrix saved at: D:\Data_Science\Append_Data_Engineer_AWS_MLOPS\Data_Scientist_Rakuten_Project-main\data\processed\Xval_matrix.pkl
[✔] TF-IDF test matrix saved at: D:\Data_Science\Append_Data_Engineer_AWS_MLOPS\Data_Scientist_Rakuten_Project-main\data\processed\Xtest_matrix.pkl
[✔] TF-IDF vectorizer saved at: D:\Data_Science\Append_Data_Engineer_AWS_MLOPS\Data_Scientist_Rakuten_Project-main\data\processed\tfidf_vectorizer.pkl


## 5. 🔄 Next Steps  

Now that we have **vectorized our text data using TF-IDF**, a fundamental step in text-based Machine Learning, we are ready to move towards Deep Learning.  

###  **Next Steps in Our Pipeline**  
 
**1. Train and evaluate ML models using TF-IDF features** → Apply classifiers like Logistic Regression, Random Forest, and XGBoost, and tune hyperparameters on `Xval_matrix.pkl` to select the best model.  
**2. Prepare text data for Deep Learning** → Tokenization and sequencing to make the dataset compatible with neural networks.  

➡️ *Continue with the next notebook:*  
**`08_DL_Text_Tokenization_and_Sequencing.ipynb`**
